# Homogeneous Planck white-noise mocks

NPIPE noise under

`/rds/rds-lxu/flamingo/integrated_maps_synthetic/planck_noise/npipe/`

is **spatially inhomogeneous** (scanning-depth anisotropy + $1/f$ residuals). This notebook builds matching **homogeneous white-noise** realisations from the Planck channel table (beam FWHM, $w^{-1/2}$ in $\mu\mathrm{K}\,\mathrm{arcmin}$, and the implied flat $N_\ell$) and stores them at

`/rds/rds-lxu/flamingo/integrated_maps_synthetic/planck_noise/homogeneous/{freq}GHz/`

Then it compares full-sky $C_\ell$ of the homogeneous mocks against the on-disk NPIPE A-split maps (HFI 100–857 GHz).

**What is generated**

| | |
|---|---|
| Frequencies | 30, 44, 70, 100, 143, 217, 353, 545, 857 GHz |
| $N_\mathrm{side}$ | 2048 (same as NPIPE) |
| Units | $\mu\mathrm{K}_\mathrm{CMB}$, RING, Galactic |
| Beam | **not** applied — table FWHM is recorded in the FITS header only |
| Realisations | one independent Gaussian map per frequency |

White noise is drawn in **pixel space** with
$\sigma_\mathrm{pix} = (w^{-1/2}) / \sqrt{\Omega_\mathrm{pix}}$,
which gives a flat harmonic spectrum $C_\ell \approx N_\ell$ on the pixelised map
(`anafast` without pixel-window deconvolution — that deconvolution over-corrects
pixel-space white noise), where

$$N_\ell = \left(w^{-1/2}\cdot\frac{\pi}{180\cdot 60}\right)^2.$$

NPIPE comparison uses detector-set **A**, `mc_00200`. The table is a **full-mission** white-noise level, so an A-split can sit near $\sim 2\,N_\ell$ at high $\ell$. LFI (30/44/70) has no NPIPE files on disk here; those channels are generated but not compared.


## 1. Configuration

In [1]:
from pathlib import Path

import healpy as hp
import matplotlib.pyplot as plt
import numpy as np

from flamingo_mock.io import write_map
from flamingo_mock.powerspectra import bin_cl, compute_cl
from flamingo_mock.spectral import intensity_to_uK

NSIDE = 2048
SEED = 42
WRITE_MAPS = True          # skip existing files unless OVERWRITE
OVERWRITE = False
RECOMPUTE_CL = False       # True forces a fresh anafast pass
LMAX = 2 * NSIDE           # 4096
DELTA_ELL = 21
NPIPE_REAL = 200
NPIPE_SPLIT = "A"

OUT_ROOT = Path("/rds/rds-lxu/flamingo/integrated_maps_synthetic") / "planck_noise" / "homogeneous"
NPIPE_ROOT = Path("/rds/rds-lxu/flamingo/integrated_maps_synthetic") / "planck_noise" / "npipe"
FIG_DIR = Path("../figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)
OUT_ROOT.mkdir(parents=True, exist_ok=True)

CL_CACHE = OUT_ROOT / "spectra_white_vs_npipe.npz"

print(f"nside     : {NSIDE}")
print(f"lmax      : {LMAX}")
print(f"out       : {OUT_ROOT}")
print(f"npipe     : {NPIPE_ROOT}")
print(f"cl cache  : {CL_CACHE}")


nside     : 2048
lmax      : 4096
out       : /rds/rds-lxu/flamingo/integrated_maps_synthetic/planck_noise/homogeneous
npipe     : /rds/rds-lxu/flamingo/integrated_maps_synthetic/planck_noise/npipe
cl cache  : /rds/rds-lxu/flamingo/integrated_maps_synthetic/planck_noise/homogeneous/spectra_white_vs_npipe.npz


## 2. Planck white-noise table

$w^{-1/2}$ is the map-level white-noise RMS in $\mu\mathrm{K}\,\mathrm{arcmin}$.
$N_\ell$ follows by converting arcmin $\to$ radians and squaring. Beam FWHMs are **not** applied to the mocks.


In [2]:
# Frequency, FWHM [arcmin], w^{-1/2} [uK arcmin], N_ell [uK^2]
TABLE = {
    30:  {"fwhm": 32.29, "uk_arcmin": 150.0,  "nell": 0.00190},
    44:  {"fwhm": 27.94, "uk_arcmin": 162.0,  "nell": 0.00222},
    70:  {"fwhm": 13.08, "uk_arcmin": 210.0,  "nell": 0.00373},
    100: {"fwhm": 9.66,  "uk_arcmin": 77.4,   "nell": 0.000507},
    143: {"fwhm": 7.22,  "uk_arcmin": 33.0,   "nell": 9.21e-5},
    217: {"fwhm": 4.90,  "uk_arcmin": 46.8,   "nell": 0.000185},
    353: {"fwhm": 4.92,  "uk_arcmin": 154.0,  "nell": 0.00200},
    545: {"fwhm": 4.67,  "uk_arcmin": 806.7,  "nell": 0.0551},
    857: {"fwhm": 4.22,  "uk_arcmin": 19115.0,"nell": 30.9},
}
FREQS = tuple(TABLE)
NPIPE_FREQS = (100, 143, 217, 353, 545, 857)

ARCMIN_TO_RAD = np.pi / (180.0 * 60.0)
OMEGA_ARCMIN2 = hp.nside2pixarea(NSIDE, degrees=True) * 3600.0

print(f"{'nu':>5}  {'FWHM':>6}  {'w^-1/2':>10}  {'N_ell table':>12}  "
      f"{'N_ell from w':>12}  {'sigma_pix':>10}")
print(f"{'GHz':>5}  {'arcmin':>6}  {'uK arcmin':>10}  {'uK^2':>12}  "
      f"{'uK^2':>12}  {'uK':>10}")
for nu, row in TABLE.items():
    nell_from_w = (row["uk_arcmin"] * ARCMIN_TO_RAD) ** 2
    sig = row["uk_arcmin"] / np.sqrt(OMEGA_ARCMIN2)
    print(f"{nu:5d}  {row['fwhm']:6.2f}  {row['uk_arcmin']:10.1f}  "
          f"{row['nell']:12.4e}  {nell_from_w:12.4e}  {sig:10.3f}")
print(f"\npixel area at nside={NSIDE}: {OMEGA_ARCMIN2:.4f} arcmin^2")


   nu    FWHM      w^-1/2   N_ell table  N_ell from w   sigma_pix
  GHz  arcmin   uK arcmin          uK^2          uK^2          uK
   30   32.29       150.0    1.9000e-03    1.9039e-03      87.324
   44   27.94       162.0    2.2200e-03    2.2207e-03      94.310
   70   13.08       210.0    3.7300e-03    3.7316e-03     122.253
  100    9.66        77.4    5.0700e-04    5.0691e-04      45.059
  143    7.22        33.0    9.2100e-05    9.2147e-05      19.211
  217    4.90        46.8    1.8500e-04    1.8533e-04      27.245
  353    4.92       154.0    2.0000e-03    2.0068e-03      89.653
  545    4.67       806.7    5.5100e-02    5.5065e-02     469.628
  857    4.22     19115.0    3.0900e+01    3.0917e+01   11127.973

pixel area at nside=2048: 2.9506 arcmin^2


## 3. Draw and write homogeneous maps

Independent seeds per frequency (`SEED + nu`) so the channels are uncorrelated.
Maps are `float32`, RING, $\mu\mathrm{K}_\mathrm{CMB}$.


In [3]:
def homog_path(nu: int) -> Path:
    return OUT_ROOT / f"{nu}GHz" / f"white_noise_{nu}GHz_nside{NSIDE}_uK.fits"


def make_white_noise_map(nu: int, rng: np.random.Generator) -> np.ndarray:
    sigma_pix = TABLE[nu]["uk_arcmin"] / np.sqrt(OMEGA_ARCMIN2)
    return rng.normal(0.0, sigma_pix, hp.nside2npix(NSIDE))


written = []
skipped = []
for nu in FREQS:
    dest = homog_path(nu)
    if dest.is_file() and not OVERWRITE:
        skipped.append(nu)
        print(f"skip     {nu} GHz  <- {dest}")
        continue
    if not WRITE_MAPS:
        print(f"dry-run  {nu} GHz  -> {dest}")
        continue
    rng = np.random.default_rng(SEED + int(nu))
    m = make_white_noise_map(nu, rng)
    extra = [
        ("FWHM", TABLE[nu]["fwhm"], "beam FWHM [arcmin], NOT applied"),
        ("UKARCMIN", TABLE[nu]["uk_arcmin"], "white noise w^{-1/2} [uK arcmin]"),
        ("NELL", TABLE[nu]["nell"], "target white N_ell [uK^2]"),
        ("SEED", SEED + int(nu), "np.random.default_rng seed"),
        ("COMMENT", "homogeneous pixel white noise; no beam smoothing"),
    ]
    write_map(dest, m, unit="uK_CMB", freq=float(nu), extra=extra, dtype=np.float32)
    written.append(nu)

print(f"\nwrote {len(written)}  skipped {len(skipped)}")


skip     30 GHz  <- /rds/rds-lxu/flamingo/integrated_maps_synthetic/planck_noise/homogeneous/30GHz/white_noise_30GHz_nside2048_uK.fits
skip     44 GHz  <- /rds/rds-lxu/flamingo/integrated_maps_synthetic/planck_noise/homogeneous/44GHz/white_noise_44GHz_nside2048_uK.fits
skip     70 GHz  <- /rds/rds-lxu/flamingo/integrated_maps_synthetic/planck_noise/homogeneous/70GHz/white_noise_70GHz_nside2048_uK.fits
skip     100 GHz  <- /rds/rds-lxu/flamingo/integrated_maps_synthetic/planck_noise/homogeneous/100GHz/white_noise_100GHz_nside2048_uK.fits
skip     143 GHz  <- /rds/rds-lxu/flamingo/integrated_maps_synthetic/planck_noise/homogeneous/143GHz/white_noise_143GHz_nside2048_uK.fits
skip     217 GHz  <- /rds/rds-lxu/flamingo/integrated_maps_synthetic/planck_noise/homogeneous/217GHz/white_noise_217GHz_nside2048_uK.fits
skip     353 GHz  <- /rds/rds-lxu/flamingo/integrated_maps_synthetic/planck_noise/homogeneous/353GHz/white_noise_353GHz_nside2048_uK.fits
skip     545 GHz  <- /rds/rds-lxu/flamingo/

## 4. Load maps in $\mu\mathrm{K}_\mathrm{CMB}$

NPIPE 100–353 GHz is $\mathrm{K}_\mathrm{CMB}$ ($\times 10^6$).
545 and 857 GHz are $\mathrm{MJy}/\mathrm{sr}$ and go through `intensity_to_uK`.


In [4]:
def npipe_path(nu: int, split: str = NPIPE_SPLIT, real: int = NPIPE_REAL) -> Path:
    name = f"npipe6v20_noise_{nu}_{split}_mc_{real:05d}.fits"
    return NPIPE_ROOT / f"{nu}GHz" / split / name


def load_homog_uK(nu: int) -> np.ndarray:
    p = homog_path(nu)
    if not p.is_file():
        raise FileNotFoundError(p)
    return np.asarray(hp.read_map(str(p), field=0, dtype=np.float64))


def load_npipe_uK(nu: int) -> np.ndarray:
    p = npipe_path(nu)
    if not p.is_file():
        raise FileNotFoundError(p)
    m = np.asarray(hp.read_map(str(p), field=0, dtype=np.float64))
    if nu <= 353:
        return m * 1.0e6
    return intensity_to_uK(m * 1.0e6, float(nu))


print(f"{'nu':>5}  {'homog rms':>12}  {'sigma_pix':>12}  {'NPIPE A rms':>12}  {'NPIPE file'}")
for nu in FREQS:
    mh = load_homog_uK(nu)
    sig = TABLE[nu]["uk_arcmin"] / np.sqrt(OMEGA_ARCMIN2)
    p_npipe = npipe_path(nu)
    if p_npipe.is_file():
        mn = load_npipe_uK(nu)
        print(f"{nu:5d}  {mh.std():12.3f}  {sig:12.3f}  {mn.std():12.3f}  {p_npipe.name}")
        del mn
    else:
        print(f"{nu:5d}  {mh.std():12.3f}  {sig:12.3f}  {'(none)':>12}  —")
    del mh


   nu     homog rms     sigma_pix   NPIPE A rms  NPIPE file


   30        87.317        87.324        (none)  —
   44        94.301        94.310        (none)  —


   70       122.251       122.253        (none)  —


  100        45.059        45.059        79.694  npipe6v20_noise_100_A_mc_00200.fits


  143        19.213        19.211        24.861  npipe6v20_noise_143_A_mc_00200.fits


  217        27.243        27.245        42.550  npipe6v20_noise_217_A_mc_00200.fits


  353        89.668        89.653       186.195  npipe6v20_noise_353_A_mc_00200.fits


  545       469.606       469.628        56.687  npipe6v20_noise_545_A_mc_00200.fits


  857     11125.453     11127.973    159773.052  npipe6v20_noise_857_A_mc_00200.fits


## 5. Power spectra

Full-sky `anafast`, monopole subtracted, **no** pixel-window deconvolution.
Pixel-space white noise is already discrete; dividing by $p_\ell^2$ over-corrects
and produces a spurious high-$\ell$ upturn. Homogeneous $C_\ell$ should sit on
the table $N_\ell$. NPIPE is not white: excess at low $\ell$ from $1/f$ / striping,
and a high-$\ell$ plateau set by the (inhomogeneous) white level of the A-split.


In [5]:
def load_or_compute_cls():
    if CL_CACHE.is_file() and not RECOMPUTE_CL:
        data = np.load(CL_CACHE)
        print(f"loaded cache {CL_CACHE}")
        return {k: data[k] for k in data.files}

    out = {"ell": np.arange(LMAX + 1)}
    for nu in FREQS:
        print(f"anafast homogeneous {nu} GHz ...", flush=True)
        mh = load_homog_uK(nu)
        out[f"cl_homog_{nu}"] = compute_cl(
            mh, lmax=LMAX, iter=0, deconv_pixel_window=False
        )
        del mh
        if nu in NPIPE_FREQS and npipe_path(nu).is_file():
            print(f"anafast NPIPE {NPIPE_SPLIT} {nu} GHz ...", flush=True)
            mn = load_npipe_uK(nu)
            out[f"cl_npipe_{nu}"] = compute_cl(
                mn, lmax=LMAX, iter=0, deconv_pixel_window=False
            )
            del mn
    np.savez(CL_CACHE, **out)
    print(f"wrote {CL_CACHE}")
    return out


cls = load_or_compute_cls()
ell = cls["ell"]
print("keys:", sorted(cls))


anafast homogeneous 30 GHz ...


anafast homogeneous 44 GHz ...


anafast homogeneous 70 GHz ...


anafast homogeneous 100 GHz ...


anafast NPIPE A 100 GHz ...


...

anafast homogeneous 545 GHz ...


anafast NPIPE A 545 GHz ...


anafast homogeneous 857 GHz ...


anafast NPIPE A 857 GHz ...


wrote /rds/rds-lxu/flamingo/integrated_maps_synthetic/planck_noise/homogeneous/spectra_white_vs_npipe.npz
keys: ['cl_homog_100', 'cl_homog_143', 'cl_homog_217', 'cl_homog_30', 'cl_homog_353', 'cl_homog_44', 'cl_homog_545', 'cl_homog_70', 'cl_homog_857', 'cl_npipe_100', 'cl_npipe_143', 'cl_npipe_217', 'cl_npipe_353', 'cl_npipe_545', 'cl_npipe_857', 'ell']


## 6. High-$\ell$ plateau vs table $N_\ell$

In [6]:
ELL_LO, ELL_HI = 1000, 2000

rows = []
print(f"{'nu':>5}  {'N_ell':>10}  {'homog':>10}  {'h/N':>7}  "
      f"{'NPIPE A':>10}  {'n/N':>7}  {'n/(2N)':>7}")
for nu in FREQS:
    nell = TABLE[nu]["nell"]
    h = cls[f"cl_homog_{nu}"]
    h_plat = np.nanmedian(h[ELL_LO:ELL_HI + 1])
    key = f"cl_npipe_{nu}"
    if key in cls:
        n = cls[key]
        n_plat = np.nanmedian(n[ELL_LO:ELL_HI + 1])
        print(f"{nu:5d}  {nell:10.3e}  {h_plat:10.3e}  {h_plat/nell:7.3f}  "
              f"{n_plat:10.3e}  {n_plat/nell:7.3f}  {n_plat/(2*nell):7.3f}")
    else:
        print(f"{nu:5d}  {nell:10.3e}  {h_plat:10.3e}  {h_plat/nell:7.3f}  "
              f"{'—':>10}  {'—':>7}  {'—':>7}")
print(f"\nPlateau is the median C_ell over {ELL_LO} <= ell <= {ELL_HI}.")
print("NPIPE n/(2N) ~ 1 is the naive A-split expectation (half the hits).")


   nu       N_ell       homog      h/N     NPIPE A      n/N   n/(2N)
   30   1.900e-03   1.907e-03    1.004           —        —        —
   44   2.220e-03   2.217e-03    0.999           —        —        —
   70   3.730e-03   3.723e-03    0.998           —        —        —
  100   5.070e-04   5.069e-04    1.000   8.366e-04    1.650    0.825
  143   9.210e-05   9.205e-05    0.999   1.599e-04    1.736    0.868
  217   1.850e-04   1.852e-04    1.001   7.754e-04    4.191    2.096
  353   2.000e-03   2.009e-03    1.004   3.555e-02   17.774    8.887
  545   5.510e-02   5.509e-02    1.000   4.251e-03    0.077    0.039
  857   3.090e+01   3.092e+01    1.001   4.049e+04  1310.477  655.238

Plateau is the median C_ell over 1000 <= ell <= 2000.
NPIPE n/(2N) ~ 1 is the naive A-split expectation (half the hits).


Homogeneous / table $N_\ell$ is $1.000\pm0.004$ in every band — the mocks are on target.

NPIPE A is **not** expected to match $N_\ell$ exactly:

- The table is a full-mission white-noise level; an A-split can sit near $\sim 2N_\ell$.
- NPIPE is a mapmaking residual ($1/f$, striping, leftover systematics), not pixel-white noise.
- 100 and 143 GHz high-$\ell$ plateaus are $\sim 1.7\,N_\ell$, consistent with an A-split.
- 217 and 353 GHz stay well above $N_\ell$ even at $\ell\sim10^3$ (correlated residual power).
- 545 GHz NPIPE (converted $\mathrm{MJy}/\mathrm{sr}\to\mu\mathrm{K}$) is *below* the table; 857 GHz is far above — both are residual maps, not the Planck 2018 white-noise forecast. Compare shapes, not a single ratio.

## 7. $C_\ell$ comparison (HFI, vs NPIPE)

Solid: homogeneous mock. Dashed: NPIPE A `mc_00200`. Dotted: table $N_\ell$. Dash-dot: $2N_\ell$ (A-split white-noise guide).


In [7]:
fig, axes = plt.subplots(2, 3, figsize=(12.5, 7.5), sharex=True, sharey=False)
axes = axes.ravel()
for ax, nu in zip(axes, NPIPE_FREQS):
    nell = TABLE[nu]["nell"]
    eh, ch = bin_cl(cls[f"cl_homog_{nu}"], delta_ell=DELTA_ELL, lmin=2)
    ax.loglog(eh, ch, color="C0", lw=1.6, label="homogeneous")
    key = f"cl_npipe_{nu}"
    if key in cls:
        en, cn = bin_cl(cls[key], delta_ell=DELTA_ELL, lmin=2)
        ax.loglog(en, cn, color="C3", lw=1.4, ls="--", label=f"NPIPE {NPIPE_SPLIT}")
    ax.axhline(nell, color="k", ls=":", lw=1.1, label=r"table $N_\ell$")
    ax.axhline(2.0 * nell, color="0.45", ls="-.", lw=1.0, label=r"$2N_\ell$ (A-split)")
    ax.set_title(f"{nu} GHz")
    ax.set_xlim(2, LMAX)
    ax.set_ylabel(r"$C_\ell$ [$\mu\mathrm{K}^2$]")
axes[0].legend(loc="upper right", fontsize=8, frameon=False)
for ax in axes[3:]:
    ax.set_xlabel(r"Multipole $\ell$")
fig.suptitle("Homogeneous white noise vs NPIPE", y=1.01)
fig.tight_layout()
out = FIG_DIR / "homogeneous_vs_npipe_cl_hfi.png"
fig.savefig(out, dpi=150, bbox_inches="tight")
print("wrote", out)
plt.show()


wrote ../figures/homogeneous_vs_npipe_cl_hfi.png


## 8. Homogeneous mocks vs table $N_\ell$ (all nine channels)


In [8]:
fig, axes = plt.subplots(3, 3, figsize=(12.5, 10.0), sharex=True)
axes = axes.ravel()
for ax, nu in zip(axes, FREQS):
    nell = TABLE[nu]["nell"]
    eh, ch = bin_cl(cls[f"cl_homog_{nu}"], delta_ell=DELTA_ELL, lmin=2)
    ax.loglog(eh, ch, color="C0", lw=1.5, label="homogeneous")
    ax.axhline(nell, color="k", ls=":", lw=1.1, label=r"table $N_\ell$")
    ax.set_title(f"{nu} GHz")
    ax.set_xlim(2, LMAX)
    ax.set_ylabel(r"$C_\ell$ [$\mu\mathrm{K}^2$]")
axes[0].legend(loc="upper right", fontsize=8, frameon=False)
for ax in axes[6:]:
    ax.set_xlabel(r"Multipole $\ell$")
fig.suptitle("Homogeneous white-noise mocks vs table $N_\ell$", y=1.01)
fig.tight_layout()
out = FIG_DIR / "homogeneous_white_noise_cl_allfreq.png"
fig.savefig(out, dpi=150, bbox_inches="tight")
print("wrote", out)
plt.show()


<>:14: SyntaxWarning: invalid escape sequence '\e'
<>:14: SyntaxWarning: invalid escape sequence '\e'
/tmp/ipykernel_2030152/2426504780.py:14: SyntaxWarning: invalid escape sequence '\e'
  fig.suptitle("Homogeneous white-noise mocks vs table $N_\ell$", y=1.01)


wrote ../figures/homogeneous_white_noise_cl_allfreq.png


## 9. Map gallery — homogeneous vs NPIPE at 143 GHz

Same colour scale. NPIPE shows ecliptic-plane scanning stripes; the mock is statistically uniform.


In [9]:
nu = 143
mh = load_homog_uK(nu)
mn = load_npipe_uK(nu)
vmax = np.percentile(np.abs(np.concatenate([mh, mn])), 98)

fig = plt.figure(figsize=(12.5, 4.6))
hp.mollview(mh, fig=fig.number, sub=(1, 2, 1), min=-vmax, max=vmax,
            title=f"{nu} GHz homogeneous white noise",
            unit=r"$\mu\mathrm{K}$", cmap="RdBu_r", notext=True)
hp.mollview(mn, fig=fig.number, sub=(1, 2, 2), min=-vmax, max=vmax,
            title=f"{nu} GHz NPIPE {NPIPE_SPLIT}  mc_{NPIPE_REAL:05d}",
            unit=r"$\mu\mathrm{K}$", cmap="RdBu_r", notext=True)
out = FIG_DIR / f"homogeneous_vs_npipe_{nu}GHz_mollview.png"
fig.savefig(out, dpi=150, bbox_inches="tight")
print("wrote", out)
print(f"homog  rms={mh.std():.2f} uK   NPIPE rms={mn.std():.2f} uK")
plt.show()


wrote ../figures/homogeneous_vs_npipe_143GHz_mollview.png
homog  rms=19.21 uK   NPIPE rms=24.86 uK


## 10. Output inventory


In [10]:
print("Homogeneous maps")
for nu in FREQS:
    p = homog_path(nu)
    if p.is_file():
        print(f"  {nu:3d} GHz  {p.stat().st_size/1e6:7.1f} MB  {p}")
    else:
        print(f"  {nu:3d} GHz  MISSING  {p}")
print()
print("NPIPE comparison maps")
for nu in NPIPE_FREQS:
    p = npipe_path(nu)
    flag = "ok" if p.is_file() else "MISSING"
    print(f"  {nu:3d} GHz  {flag:7s}  {p}")
print()
print("figures:")
for name in (
    "homogeneous_vs_npipe_cl_hfi.png",
    "homogeneous_white_noise_cl_allfreq.png",
    "homogeneous_vs_npipe_143GHz_mollview.png",
):
    p = FIG_DIR / name
    print(f"  {'ok' if p.is_file() else 'MISSING'}  {p.resolve()}")


Homogeneous maps
   30 GHz    201.3 MB  /rds/rds-lxu/flamingo/integrated_maps_synthetic/planck_noise/homogeneous/30GHz/white_noise_30GHz_nside2048_uK.fits
   44 GHz    201.3 MB  /rds/rds-lxu/flamingo/integrated_maps_synthetic/planck_noise/homogeneous/44GHz/white_noise_44GHz_nside2048_uK.fits
   70 GHz    201.3 MB  /rds/rds-lxu/flamingo/integrated_maps_synthetic/planck_noise/homogeneous/70GHz/white_noise_70GHz_nside2048_uK.fits
  100 GHz    201.3 MB  /rds/rds-lxu/flamingo/integrated_maps_synthetic/planck_noise/homogeneous/100GHz/white_noise_100GHz_nside2048_uK.fits
  143 GHz    201.3 MB  /rds/rds-lxu/flamingo/integrated_maps_synthetic/planck_noise/homogeneous/143GHz/white_noise_143GHz_nside2048_uK.fits
  217 GHz    201.3 MB  /rds/rds-lxu/flamingo/integrated_maps_synthetic/planck_noise/homogeneous/217GHz/white_noise_217GHz_nside2048_uK.fits
  353 GHz    201.3 MB  /rds/rds-lxu/flamingo/integrated_maps_synthetic/planck_noise/homogeneous/353GHz/white_noise_353GHz_nside2048_uK.fits
  545 GHz